# 02 — Opening Inventory

Generates opening stock balances per product per branch.
Requires `01_master_data` to have been run first.

In [ ]:
%run ./00_helpers

In [ ]:
RANDOM_SEED          = 42
SCHEMA_NAME          = "rockline"
SNAPSHOT_DATE        = ""
HISTORY_START_DATE   = "2022-01-01"
HUB_STOCK_MULTIPLIER = 3.0

In [ ]:
import random
from datetime import datetime, date, timedelta
from decimal import Decimal

rng = random.Random(RANDOM_SEED)
_NOW = datetime.utcnow()

if SNAPSHOT_DATE:
    snap_date = date.fromisoformat(SNAPSHOT_DATE)
else:
    snap_date = date.fromisoformat(HISTORY_START_DATE) - timedelta(days=1)

print(f"Snapshot date: {snap_date}")

## Load Reference Data

In [ ]:
products_pd  = spark.table(f"{SCHEMA_NAME}.dim_product").toPandas()
branches_pd  = spark.table(f"{SCHEMA_NAME}.dim_branch").toPandas()
print(f"Loaded {len(products_pd)} products, {len(branches_pd)} branches")

## Generate Inventory Snapshot

In [ ]:
_OPENING_QTY_RANGES = {
    1: (50,  500),   # CC: bags / m3
    2: (20,  200),   # ST: linear metres / sheets
    3: (50,  300),   # TM: sheets / linear metres
    4: (20,  200),   # RF: tiles / rolls
    5: (100, 2000),  # MA: bricks / blocks (high unit count)
    6: (10,  100),   # PD: pipes / fittings
    7: (20,  200),   # EL: cable metres / units
    8: (50,  500),   # FF: boxes / units
}

snapshot_rows  = []
movement_rows  = []
snap_id = 1
mov_id  = 1
snap_ts = datetime.combine(snap_date, datetime.min.time())

for _, prod in products_pd.iterrows():
    pid  = int(prod["product_id"])
    cid  = int(prod["category_id"])
    cost = float(prod["standard_cost_gbp"])
    hub_only = bool(prod["is_stocked_at_hub_only"])
    lo, hi = _OPENING_QTY_RANGES.get(cid, (20, 100))

    for _, branch in branches_pd.iterrows():
        bid = int(branch["branch_id"])
        if hub_only and bid != 1:
            continue
        base_qty = Decimal(str(round(rng.uniform(lo, hi), 3)))
        qty = Decimal(str(round(float(base_qty) * (HUB_STOCK_MULTIPLIER if bid == 1 else 1.0), 3)))
        reorder = Decimal(str(round(float(qty) * 0.2, 3)))
        max_lvl = Decimal(str(round(float(qty) * 1.5, 3)))

        snapshot_rows.append((
            snap_id, pid, bid, snap_date,
            qty, Decimal("0.000"), qty,
            reorder, max_lvl,
            Decimal(str(round(cost, 4))),
            _NOW,
        ))

        movement_rows.append((
            mov_id, pid, bid, snap_ts,
            "opening_balance", None, None,
            qty, Decimal(str(round(cost, 4))),
            _NOW,
        ))

        snap_id += 1
        mov_id  += 1

print(f"Generated {len(snapshot_rows)} snapshot rows, {len(movement_rows)} movement rows")

## Write Tables

In [ ]:
snap_df = to_spark_df(snapshot_rows, SCHEMA_FACT_INVENTORY_SNAPSHOT)
snap_df.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMA_NAME}.fact_inventory_snapshot")
print(f"fact_inventory_snapshot: {snap_df.count()} rows")

mov_df = to_spark_df(movement_rows, SCHEMA_FACT_INVENTORY_MOVEMENT)
mov_df.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMA_NAME}.fact_inventory_movement")
print(f"fact_inventory_movement: {mov_df.count()} rows")